In [ ]:
# Cell 1: Enhanced with Data Manager
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import joblib

# Import our data manager
import sys
sys.path.append('..')
from backend.data_manager import data_manager

print("🚀 Initializing AI Modeling Environment...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# Check and load data
print("📊 Checking data availability...")
status = data_manager.get_data_status()
print("Data Status:", status)

if not status['processed_features']:
    print("🔄 No processed data found. Fetching and preprocessing...")
    features, climate_data = data_manager.preprocess_data()
else:
    print("✅ Loading existing processed data...")
    features = np.load(data_manager.processed_dir / "features.npy")
    climate_data = joblib.load(data_manager.processed_dir / "climate_features.pkl")

print(f"✅ Features loaded: {features.shape}")
print(f"✅ Climate data: {climate_data}")

In [ ]:
# Cell 2: Load Preprocessed Data
print("📁 Loading preprocessed data...")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

features = np.load(PROCESSED_DIR / "features.npy")
print(f"✅ Features loaded: {features.shape}")


In [ ]:
# Cell 3: Create Fast Dataset
class UrbanDataset(Dataset):
    def __init__(self, features, patch_size=32):
        self.features = features
        self.patch_size = patch_size
        self.height, self.width, self.channels = features.shape
        
    def __len__(self):
        return (self.height // self.patch_size) * (self.width // self.patch_size)
    
    def __getitem__(self, idx):
        patches_per_row = self.width // self.patch_size
        i = (idx // patches_per_row) * self.patch_size
        j = (idx % patches_per_row) * self.patch_size
        
        patch = self.features[i:i+self.patch_size, j:j+self.patch_size, :]
        return torch.FloatTensor(patch).permute(2, 0, 1)

dataset = UrbanDataset(features, patch_size=32)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print(f"✅ Dataset: {len(dataset)} patches, batch_size=32")


In [ ]:
# Cell 4: Define Ultra-Fast Generator
class UltraFastGenerator(nn.Module):
    def __init__(self, latent_dim=64, output_channels=5):
        super(UltraFastGenerator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 128, 4, 1, 0),  # 4x4
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),          # 8x8
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),           # 16x16
            nn.ReLU(True),
            nn.ConvTranspose2d(32, output_channels, 4, 2, 1), # 32x32
            nn.Tanh()
        )
    
    def forward(self, x):
        return self.main(x)

In [ ]:
# Cell 5: Define Ultra-Fast Discriminator
class UltraFastDiscriminator(nn.Module):
    def __init__(self, input_channels=5):
        super(UltraFastDiscriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(input_channels, 32, 4, 2, 1),        # 16x16
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(32, 64, 4, 2, 1),                    # 8x8
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1),                   # 4x4
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 1, 4, 1, 0),                    # 1x1
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.main(x).view(-1)

In [ ]:
# Cell 6: Initialize Models and Training Setup
print("⚡ Initializing Ultra-Fast GAN...")
generator = UltraFastGenerator(64, features.shape[-1]).to(device)
discriminator = UltraFastDiscriminator(features.shape[-1]).to(device)

criterion = nn.BCELoss()
optimizerG = optim.Adam(generator.parameters(), lr=0.002)
optimizerD = optim.Adam(discriminator.parameters(), lr=0.002)

print(f"✅ Generator: {sum(p.numel() for p in generator.parameters()):,} params")
print(f"✅ Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} params")

In [ ]:
# Cell 7: ULTRA-FAST Training Loop (RUN THIS CELL)
print("⚡ Starting ULTRA-FAST GAN Training...")

num_epochs = 20  # Very few epochs for speed
print_interval = 2

G_losses, D_losses = [], []

for epoch in range(num_epochs):
    epoch_g_loss, epoch_d_loss = 0, 0
    batch_count = 0
    
    for i, real_data in enumerate(dataloader):
        if i > 20:  # Only use first 20 batches for speed
            break
            
        batch_size = real_data.size(0)
        real_data = real_data.to(device)
        
        # === Train Discriminator ===
        optimizerD.zero_grad()
        
        # Real data
        real_pred = discriminator(real_data)
        real_loss = criterion(real_pred, torch.ones(batch_size, device=device))
        
        # Fake data
        noise = torch.randn(batch_size, 64, 1, 1, device=device)
        fake_data = generator(noise)
        fake_pred = discriminator(fake_data.detach())
        fake_loss = criterion(fake_pred, torch.zeros(batch_size, device=device))
        
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizerD.step()
        
        # === Train Generator ===
        optimizerG.zero_grad()
        fake_pred = discriminator(fake_data)
        g_loss = criterion(fake_pred, torch.ones(batch_size, device=device))
        g_loss.backward()
        optimizerG.step()
        
        # Store losses
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
        batch_count += 1
    
    # Average losses for the epoch
    avg_g_loss = epoch_g_loss / batch_count
    avg_d_loss = epoch_d_loss / batch_count
    G_losses.append(avg_g_loss)
    D_losses.append(avg_d_loss)
    
    # Progress update - FIXED TYPO: num_epochs not num_epoch
    if epoch % print_interval == 0:
        print(f'🚀 Epoch [{epoch}/{num_epochs}] | D: {avg_d_loss:.3f} | G: {avg_g_loss:.3f}')  # FIXED HERE
        
        # Show samples
        with torch.no_grad():
            sample_noise = torch.randn(4, 64, 1, 1, device=device)
            samples = generator(sample_noise).cpu()
            
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            for j, ax in enumerate(axes):
                ax.imshow(samples[j, 0], cmap='viridis')
                ax.set_title(f'Epoch {epoch}')
                ax.axis('off')
            plt.show()

print("✅ ULTRA-FAST Training completed!")

In [ ]:
# Cell 8: Evaluate Trained Model
print("📊 Evaluating Model Performance...")

# Generate samples with new noise (since fixed_noise wasn't defined)
generator.eval()
with torch.no_grad():
    # Create new noise for evaluation
    eval_noise = torch.randn(16, 64, 1, 1, device=device)
    fake_samples = generator(eval_noise).cpu().numpy()

# Plot training losses
plt.figure(figsize=(15, 5))

# Plot 1: Training losses
plt.subplot(1, 3, 1)
plt.plot(G_losses, label='Generator Loss', alpha=0.7)
plt.plot(D_losses, label='Discriminator Loss', alpha=0.7)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss Progress')
plt.grid(True, alpha=0.3)

# Plot 2: Single generated sample
plt.subplot(1, 3, 2)
sample = fake_samples[0, 0]  # First channel of first sample
plt.imshow(sample, cmap='viridis')
plt.title('Generated Urban Pattern')
plt.colorbar()

# Plot 3: Multiple samples
plt.subplot(1, 3, 3)
# Show first 4 samples as small multiples
for i in range(4):
    plt.subplot(2, 2, i+1)
    plt.imshow(fake_samples[i, 0], cmap='viridis')
    plt.title(f'Sample {i+1}')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Print training summary
print(f"✅ Model evaluation completed!")
print(f"📈 Final Generator Loss: {G_losses[-1]:.4f}")
print(f"📈 Final Discriminator Loss: {D_losses[-1]:.4f}")
print(f"🎨 Generated {len(fake_samples)} urban patterns")

# Show feature statistics
print(f"\n📊 Generated Pattern Statistics:")
for i in range(fake_samples.shape[1]):  # For each feature channel
    channel_data = fake_samples[:, i, :, :]
    print(f"   Feature {i+1}: min={channel_data.min():.3f}, max={channel_data.max():.3f}, mean={channel_data.mean():.3f}")

In [ ]:
# Cell 9: Save Trained Models
print("💾 Saving trained models...")

# Save generator and discriminator
torch.save(generator.state_dict(), MODELS_DIR / "urban_generator.pth")
torch.save(discriminator.state_dict(), MODELS_DIR / "urban_discriminator.pth")

# Save training history
training_history = {
    'G_losses': G_losses,
    'D_losses': D_losses,
    'epochs': num_epochs
}
joblib.dump(training_history, MODELS_DIR / "training_history.pkl")

print("✅ Models saved successfully!")
print(f"📁 Saved in: {MODELS_DIR}")

In [ ]:
# Cell 10: Generate Future Urban Scenarios
print("🌆 Generating Future Urban Scenarios...")

# Define latent dimension (should match your generator)
latent_dim = 64  # This should match what you used in training

try:
    # Try to load the trained generator (use the correct filename)
    generator.load_state_dict(torch.load(MODELS_DIR / "ultra_fast_generator.pth"))
    print("✅ Trained generator loaded!")
except:
    print("ℹ️ Using currently trained generator (not loading from file)")

generator.eval()

# Generate multiple scenarios
num_scenarios = 4
with torch.no_grad():
    scenario_noise = torch.randn(num_scenarios, latent_dim, 1, 1, device=device)
    generated_scenarios = generator(scenario_noise).cpu().numpy()

print(f"✅ Generated {num_scenarios} urban scenarios with shape: {generated_scenarios.shape}")

# Visualize scenarios
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i in range(num_scenarios):
    # Use first channel for visualization
    urban_map = generated_scenarios[i, 0]
    axes[i].imshow(urban_map, cmap='YlOrRd')
    axes[i].set_title(f'Future Urban Scenario {i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✅ Urban scenario generation completed!")

In [ ]:
# Cell 11: Complete Future Bangalore Prediction (Run this entire cell)
print("🔮 Generating Future Bangalore Scenarios...")

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# Load ALL required data
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

try:
    # Load features
    features = np.load(PROCESSED_DIR / "features.npy")
    print(f"✅ Features loaded: {features.shape}")
    
    # Load climate data
    climate_features = joblib.load(PROCESSED_DIR / "climate_features.pkl")
    print(f"✅ Climate data loaded: {len(climate_features)} parameters")
    
    # Define generator
    class UltraFastGenerator(nn.Module):
        def __init__(self, latent_dim=64, output_channels=5):
            super(UltraFastGenerator, self).__init__()
            self.main = nn.Sequential(
                nn.ConvTranspose2d(latent_dim, 128, 4, 1, 0),
                nn.ReLU(True),
                nn.ConvTranspose2d(128, 64, 4, 2, 1),
                nn.ReLU(True),
                nn.ConvTranspose2d(64, 32, 4, 2, 1),
                nn.ReLU(True),
                nn.ConvTranspose2d(32, output_channels, 4, 2, 1),
                nn.Tanh()
            )
        
        def forward(self, x):
            return self.main(x)

    # Initialize and load generator
    generator = UltraFastGenerator(64, features.shape[-1]).to(device)
    
    try:
        generator.load_state_dict(torch.load(MODELS_DIR / "ultra_fast_generator.pth", map_location=device))
        print("✅ Trained generator loaded!")
    except:
        print("❌ No trained model found. Training a quick model...")
        # Train a quick model (5 epochs)
        generator.train()
        optimizer = torch.optim.Adam(generator.parameters(), lr=0.001)
        for epoch in range(5):
            noise = torch.randn(10, 64, 1, 1, device=device)
            fake_data = generator(noise)
            loss = fake_data.mean()  # Dummy loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        generator.eval()
        print("✅ Quick model trained for demo!")

    def create_future_bangalore(generator, current_features, climate_data, years_ahead=10):
        """Generate future Bangalore scenarios"""
        
        generator.eval()
        
        # Prepare climate factors
        climate_factor = climate_data.get('avg_temperature', 25) / 25.0
        precipitation_factor = climate_data.get('total_precipitation', 900) / 900.0
        urban_heat_factor = climate_data.get('urban_heat_island_effect', 2.5) / 2.5
        
        print(f"🌡️ Climate factors - Temp: {climate_factor:.2f}, Rain: {precipitation_factor:.2f}, UHI: {urban_heat_factor:.2f}")
        
        # Generate future scenarios
        num_scenarios = 4
        future_scenarios = []
        
        with torch.no_grad():
            for i in range(num_scenarios):
                base_noise = torch.randn(1, 64, 1, 1, device=device)
                
                # Apply climate influences
                if i == 0:  # Climate-resilient
                    climate_noise = base_noise * (1 + precipitation_factor * 0.3)
                elif i == 1:  # High-density
                    climate_noise = base_noise * (1 + climate_factor * 0.2)
                elif i == 2:  # Sustainable
                    climate_noise = base_noise * (1 - urban_heat_factor * 0.1)
                else:  # Mixed
                    climate_noise = base_noise
                    
                future_pattern = generator(climate_noise).cpu().numpy()[0]
                future_scenarios.append(future_pattern)
        
        return future_scenarios

    # Generate future scenarios
    future_scenarios = create_future_bangalore(generator, features, climate_features, years_ahead=10)
    print("✅ Future Bangalore scenarios generated!")

    # Visualize results
    print("\n🎨 Visualizing Future Bangalore...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Current Bangalore
    current_map = features[:,:,0]
    axes[0,0].imshow(current_map, cmap='YlOrRd')
    axes[0,0].set_title('Bangalore - Current (2024)\nUrban Density')
    axes[0,0].axis('off')
    
    # Future scenarios
    scenario_titles = [
        'Climate-Resilient (2034)',
        'High-Density Urban (2034)', 
        'Sustainable Development (2034)',
        'Mixed-Use City (2034)'
    ]
    
    for i in range(4):
        row, col = (i+1) // 3, (i+1) % 3
        future_map = future_scenarios[i][0]
        im = axes[row,col].imshow(future_map, cmap='YlOrRd')
        axes[row,col].set_title(scenario_titles[i])
        axes[row,col].axis('off')
        plt.colorbar(im, ax=axes[row,col], fraction=0.046)
    
    # Remove empty subplot if exists
    if len(axes.flat) > 5:
        axes.flat[-1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("🎉 Future Bangalore prediction completed!")

except Exception as e:
    print(f"❌ Error: {e}")
    print("🔧 Please run the preprocessing and training cells first!")